# Setup

In [ ]:
import os
import numpy as np
import pandas as pd

os.environ["GCLOUD_PROJECT"] = "frog-id-ml"
from NatureLM.infer import Pipeline

from NatureLM.models import NatureLM
from huggingface_hub import notebook_login

In [ ]:
# Authenticate to HuggingFace
notebook_login()

In [ ]:
# Download the model from HuggingFace
model = NatureLM.from_pretrained("EarthSpeciesProject/NatureLM-audio")
model = model.eval().to("cuda")

In [ ]:
# Extract captures table
csv_path = "/home/admin/julia-dev/data/frogid_captures/FrogID_Captures_2025-03-09T21-01-13+0000.csv"
captures_df = pd.read_csv(csv_path)

captures_df

# Data Annotation for FrogID Captures

1. For each unique capture ID, run the frog vs not-a-frog query over the original and source-separated audio files
2. Take the audio file with the highest occurence of "frog" predictions to be the best candidate
3. Extract the detection windows that have been predicted as a frog and save as new audio files for training dataset

In [ ]:
import os
import pandas as pd
from pydub import AudioSegment
from tqdm import tqdm


LABEL_PATH = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/labels_29spp.csv"
label_df = pd.read_csv(LABEL_PATH)
label_df

In [ ]:
AUDIO_DIR = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp/"
PREPROCESSED_SUBDIR = "audio_29spp_preprocessed"

# Iterate over capture IDs
audio_path_dict = {}

for original_file in os.listdir(AUDIO_DIR):

    capture_id = original_file.split('.')[0]

    # Extract original and source-separated audio paths
    audio_paths = [os.path.join(AUDIO_DIR, f'{capture_id}.wav')]
    audio_paths += [os.path.join(AUDIO_DIR, PREPROCESSED_SUBDIR, f'{capture_id}_source{i}.wav') for i in list(range(4))]

    audio_path_dict[capture_id] = audio_paths

In [ ]:
query = "The objective is to classify the sound into one of the following categories: frog, birds, insects"

pipeline = Pipeline(model=model)

OUTPUT_DIR = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp/audio_29spp_annotated"

# Run the query over each capture ID
for capture_id, audio_paths in tqdm(audio_path_dict.items(), desc="Running query over capture IDs"):
    results = pipeline(audio_paths, query, window_length_seconds=5.0, hop_length_seconds=5.0)

    # Find the best candidate file based on max number of positive predictions
    frog_counts = [x.count("frog") for x in results]
    max_idx = frog_counts.index(max(frog_counts))

    best_file = audio_paths[max_idx]
    best_file_name = best_file.split('/')[-1][:-4]
    best_results = results[max_idx].split('\n')

    # Load best candidate file
    audio = AudioSegment.from_wav(best_file)
    audio = audio.set_channels(1)

    # Extract positive detection windows
    for line in best_results:
        if line == '':
            continue
        _, time_range, predicted_label = line.split('#')
        predicted_label = predicted_label[2:]
        if predicted_label != "frog":
            continue
        
        # Get start and end times
        start_time, end_time = time_range.split(' - ')
        start_time_s = int(start_time[:-4])
        end_time_s = int(end_time[:-4])
        start_time_ms, end_time_ms = start_time_s * 1000, end_time_s * 1000

        # Crop audio to detection window and save as separate file
        chunk = audio[start_time_ms:end_time_ms]
        output_path = os.path.join(OUTPUT_DIR, f"{best_file_name}_{start_time_s}s_{end_time_s}s.wav")
        chunk.export(output_path, format="wav")

        print(f"Saved audio window to: {output_path}")

# Ablation Study

## Ablation #1 - Remove frog vs not-a-frog filtering

In [ ]:
import os
import math
import pandas as pd
from pydub import AudioSegment
from tqdm import tqdm


# Load existing dataset
annotated_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio/audio_annotated"

# Extract unique filenames in dataset (ignoring time windows)
filenames = [x.split('.')[0] for x in os.listdir(annotated_audio_dir) if os.path.isfile(os.path.join(annotated_audio_dir, x))]
stems = ['_'.join(x.split('_')[:-2]) for x in filenames]
unique_stems = list(set(stems))

print(f'{len(unique_stems)} unique files')

In [ ]:
original_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio"
preprocessed_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio/audio_preprocessed"


# Load audio
output_dir = "/home/admin/julia-dev/data/frogid_captures/perch_domain_adaptation/audio/audio_chunked_no_filtering"
for filename in tqdm(unique_stems, desc="Chunking audio files"):
    filepath = (os.path.join(preprocessed_audio_dir, filename) if "source" in filename else os.path.join(original_audio_dir, filename)) + ".wav"
    audio = AudioSegment.from_wav(filepath)

    # Chunk into 5s windows
    duration_ms = len(audio)
    chunk_length_ms = 5 * 1000
    total_chunks = math.ceil(duration_ms / chunk_length_ms)

    # Export each chunk as an individual wav file
    for i in range(total_chunks):
        start_ms = i * chunk_length_ms
        end_ms = min((i + 1) * chunk_length_ms, duration_ms)
        chunk = audio[start_ms:end_ms]
        output_path = os.path.join(output_dir, f"{filename}_chunk{i}.wav")
        chunk.export(output_path, format="wav")

## Ablation #2 - Remove source separation

In [ ]:
import os
import math
import pandas as pd
from pydub import AudioSegment
from tqdm import tqdm


# Load existing dataset
annotated_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp/audio_29spp_annotated"

# Extract unique capture IDs in dataset (ignoring source ID and time windows)
capture_ids = [x.split('_')[0] for x in os.listdir(annotated_audio_dir) if os.path.isfile(os.path.join(annotated_audio_dir, x))]
unique_capture_ids = list(set(capture_ids))
print(f'{len(unique_capture_ids)} unique files')

In [ ]:
query = "The objective is to classify the sound into one of the following categories: frog, birds, insects"
pipeline = Pipeline(model=model)

original_audio_dir = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp"

audio_paths = [os.path.join(original_audio_dir, f'{x}.wav') for x in unique_capture_ids]

# Run the query over each capture ID
results = pipeline(audio_paths, query, window_length_seconds=5.0, hop_length_seconds=5.0)

results

In [ ]:
# Extract positive detection windows for each capture
output_dir = "/home/admin/julia-dev/data/frogid_captures/perch_classifier_evaluation/audio_29spp/audio_29spp_annotated_no_ss"

for i, result in enumerate(results):
    capture_id = unique_capture_ids[i]
    audio_path = audio_paths[i]
    audio = AudioSegment.from_wav(audio_path)

    for line in result.split('\n'):
        if line == '':
            continue
        _, time_range, predicted_label = line.split('#')
        predicted_label = predicted_label[2:]
        if predicted_label != "frog":
            continue
        
        # Get start and end times
        start_time, end_time = time_range.split(' - ')
        start_time_s = int(start_time[:-4])
        end_time_s = int(end_time[:-4])
        start_time_ms, end_time_ms = start_time_s * 1000, end_time_s * 1000

        # Crop audio to detection window and save as separate file
        chunk = audio[start_time_ms:end_time_ms]
        output_path = os.path.join(output_dir, f"{capture_id}_{start_time_s}s_{end_time_s}s.wav")
        chunk.export(output_path, format="wav")

        print(f"Saved audio window to: {output_path}")